In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# =========================
# CONFIG
# =========================
MODEL_NAME = "facebook/nllb-200-distilled-600M"

SRC_LANG = "npi_Deva"   # Nepali
TGT_LANG = "eng_Latn"   # English

TEXT_COL = "text"

TRAIN_PATH = "twitter_train.csv"
VAL_PATH   = "twitter_val.csv"
TEST_PATH  = "twitter_test.csv"

OUT_TRAIN_PATH = "twitter_train_en.csv"
OUT_VAL_PATH   = "twitter_val_en.csv"
OUT_TEST_PATH  = "twitter_test_en.csv"

BATCH_SIZE = 64
MAX_INPUT_LENGTH = 96
MAX_NEW_TOKENS = 96

# =========================
# LOAD MODEL
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

def clean_batch_text(batch):
    cleaned = []
    for x in batch:
        if pd.isna(x):
            cleaned.append("")
        else:
            cleaned.append(str(x).strip())
    return cleaned

def translate_text(texts, batch_size=BATCH_SIZE):
    translated_texts = []
    tokenizer.src_lang = SRC_LANG

    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i + batch_size]
        batch = clean_batch_text(batch)

        nonempty_indices = [idx for idx, txt in enumerate(batch) if txt]
        batch_outputs = [""] * len(batch)

        if nonempty_indices:
            nonempty_texts = [batch[idx] for idx in nonempty_indices]

            inputs = tokenizer(
                nonempty_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_LENGTH
            ).to(device)

            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_token_id,
                    num_beams=1,
                    do_sample=False,
                    max_new_tokens=MAX_NEW_TOKENS
                )

            decoded = tokenizer.batch_decode(
                generated_tokens,
                skip_special_tokens=True
            )

            for idx, out in zip(nonempty_indices, decoded):
                batch_outputs[idx] = out

        translated_texts.extend(batch_outputs)

    return translated_texts

def translate_file(input_path, output_path, text_col=TEXT_COL):
    print(f"\nReading: {input_path}")
    df = pd.read_csv(input_path)

    if text_col not in df.columns:
        raise ValueError(
            f"Column '{text_col}' not found in {input_path}. "
            f"Available columns: {list(df.columns)}"
        )

    texts = df[text_col].tolist()
    translated = translate_text(texts)

    df[f"{text_col}_en"] = translated
    df.to_csv(output_path, index=False, encoding="utf-8")

    print(f"Saved: {output_path}")
    print(df[[text_col, f"{text_col}_en"]].head())

translate_file(TRAIN_PATH, OUT_TRAIN_PATH)
translate_file(VAL_PATH, OUT_VAL_PATH)
translate_file(TEST_PATH, OUT_TEST_PATH)

print("\nDone. All three splits translated.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

Saved: twitter_train_en.csv
                                                text  \
0  कोरोनाको पहिलो केस आको आज ट्याक्कै महिना डिसेम...   
1                  हेटौंडाबाटै कोभिड को परीक्षण हुने   
2     कोभिड अर्घाखाँचीबाट पठाएका वटा रिपोर्ट नेगेटिभ   
3      कोभिड विश्वमहामारीको वर्णन गर्ने वटा चार्टहरु   
4  कोभिड संकट व्यवस्थापन केन्द्र सिसिएमसी ले विद्...   

                                             text_en  
0  The first case of coronavirus was reported tod...  
1      COVID testing will be conducted from Hetunda.  
2  All the reports sent from the COVID sub-region...  
3  The following charts describe the COVID pandemic.  
4  The Centre for COVID Crisis Management (CICMC)...  

Reading: twitter_val.csv


Translating: 100%|██████████| 79/79 [05:14<00:00,  3.98s/it]


Saved: twitter_val_en.csv
                                                text  \
0  भारतलाई यो कुरा हेक्का होसकी ठूलो छु भन्दैमा स...   
1  कोरोना भाइरस नेपाल गम्भीर लक्षण भएका कोभिड का ...   
2  नेपाल टेलिकमले थाल्यो कोभिड बिरुद्ध जनचेतना मो...   
3  कोभिड छलफल ग्रुपमा कोरोनाको दबाइ नै बनाउछन कि ...   
4  कालिकोटमा सूचनाको हक दावी संगठनका केन्द्रिय सद...   

                                             text_en  
0  India has to be convinced that it is big and t...  
1  Percentage of COVID patients with severe sympt...  
2  Nepal Telecom launched a message to avoid the ...  
3  The COVID discussion group is under pressure f...  
4  Tank Jais, a central member of the organizatio...  

Reading: twitter_test.csv


Translating: 100%|██████████| 79/79 [05:08<00:00,  3.91s/it]

Saved: twitter_test_en.csv
                                                text  \
0  कोभिड बाट बच्न बुद्धको यस्तो हवाई सुरक्षा निर्...   
1  कोभिड प्रतिरोध सम्मान समारोहमा चीनका राष्ट्राध...   
2  अन्य रोगको उपचार गराउन अस्पताल पुगेका र टेस्टि...   
3  मिति गतेको कोभिड बिमाको सूचनाका वारे स्पष्ट गर...   
4  पोखरा स्वास्थ्य विज्ञान प्रतिष्ठान पश्चिमाञ्चल...   

                                             text_en  
0  Such an aviation safety directory of the Buddh...  
1  Address by Chinese President Xi Jinping at a C...  
2  It is not the policy of the hospital to be tre...  
3  In connection with the clarification of the CO...  
4  Another coronavirus case has died in the Covid...  

Done. All three splits translated.


In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# =========================
# CONFIG
# =========================
MODEL_NAME = "facebook/nllb-200-distilled-600M"

SRC_LANG = "npi_Deva"   # Nepali
TGT_LANG = "eng_Latn"   # English

TEXT_COL = "text"

TRAIN_PATH = "youtube_train.csv"
VAL_PATH   = "youtube_val.csv"
TEST_PATH  = "youtube_test.csv"

OUT_TRAIN_PATH = "youtube_train_en.csv"
OUT_VAL_PATH   = "youtube_val_en.csv"
OUT_TEST_PATH  = "youtube_test_en.csv"

BATCH_SIZE = 64
MAX_INPUT_LENGTH = 96
MAX_NEW_TOKENS = 96

# =========================
# LOAD MODEL
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

def clean_batch_text(batch):
    cleaned = []
    for x in batch:
        if pd.isna(x):
            cleaned.append("")
        else:
            cleaned.append(str(x).strip())
    return cleaned

def translate_text(texts, batch_size=BATCH_SIZE):
    translated_texts = []
    tokenizer.src_lang = SRC_LANG

    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i + batch_size]
        batch = clean_batch_text(batch)

        nonempty_indices = [idx for idx, txt in enumerate(batch) if txt]
        batch_outputs = [""] * len(batch)

        if nonempty_indices:
            nonempty_texts = [batch[idx] for idx in nonempty_indices]

            inputs = tokenizer(
                nonempty_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_LENGTH
            ).to(device)

            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_token_id,
                    num_beams=1,
                    do_sample=False,
                    max_new_tokens=MAX_NEW_TOKENS
                )

            decoded = tokenizer.batch_decode(
                generated_tokens,
                skip_special_tokens=True
            )

            for idx, out in zip(nonempty_indices, decoded):
                batch_outputs[idx] = out

        translated_texts.extend(batch_outputs)

    return translated_texts

def translate_file(input_path, output_path, text_col=TEXT_COL):
    print(f"\nReading: {input_path}")
    df = pd.read_csv(input_path)

    if text_col not in df.columns:
        raise ValueError(
            f"Column '{text_col}' not found in {input_path}. "
            f"Available columns: {list(df.columns)}"
        )

    texts = df[text_col].tolist()
    translated = translate_text(texts)

    df[f"{text_col}_en"] = translated
    df.to_csv(output_path, index=False, encoding="utf-8")

    print(f"Saved: {output_path}")
    print(df[[text_col, f"{text_col}_en"]].head())

translate_file(TRAIN_PATH, OUT_TRAIN_PATH)
translate_file(VAL_PATH, OUT_VAL_PATH)
translate_file(TEST_PATH, OUT_TEST_PATH)

print("\nDone. All three YouTube splits translated.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

Saved: youtube_train_en.csv
                                                text  \
0  भ्रष्टाचारि हरु मर्नै पर्ने हो तर , अरु पनि लि...   
1  थुक् का एमाले को गोठालो यस्ता मुला जड्या दलाली...   
2  ए पुलिस को टाउके म सगँ पुलिस ज्यादती को प्रमाण...   
3  यस्ता पत्रकार लाई हटाउनु पर्छ यो के सि पनि गफड...   
4  यो मुजि पोखरेल लठौरा एक नम्बर को झुते हो साले ...   

                                             text_en  
0  If corrupt people have to die, they take other...  
1  Do not be afraid to be a good leader, but do n...  
2  There is evidence of police brutality in every...  
3  Such a journalist should be removed, even if h...  
4  This Muji Pokhreel Lathura is a number one hit...  

Reading: youtube_val.csv


Translating: 100%|██████████| 4/4 [00:02<00:00,  1.48it/s]


Saved: youtube_val_en.csv
                                                text  \
0  हत्यार बोकेर खुल्ला रूप मा हिड्ने मान्छे हरुला...   
1                 अब तपाईं अाउनु पर्याे राजनिती मा ।   
2            कोइराला को कुरा र साहस्लाइ Big salute !   
3        माओ भुक्ने मात्र हुन्न कुनै पनि काम छैन्न ।   
4  यो खाते रन्डि को छोरा कस्तो मानव अधिकार बादी हो ?   

                                             text_en  
0  Anyone who walks openly carrying a murderer mu...  
1                           You are now in politics.  
2           Speak and courage to Koirala Big salute!  
3      There is nothing more to do than just starve.  
4         What human rights defender is Randy's son?  

Reading: youtube_test.csv


Translating: 100%|██████████| 4/4 [00:02<00:00,  1.69it/s]

Saved: youtube_test_en.csv
                                                text  \
0  अगाडि बडै जानुहोस हाम्रो फुल सपोट छ र भस्टचारि...   
1  फरक बिचार का भएता पनि नेपाली राजनिति का अत्यन्...   
2  बौलाहा जस्तो अत्तो न पत्तो चिच्याउनु भन्दा सत्...   
3  जय रवि दाइ देश बिकास मा नेपाली जनता लाई सधै सा...   
4  कुनै पनि जात धर्म लाई असर नपर्ने तरिका को एउ ट...   

                                             text_en  
0  We have full support and the corruption has to...  
1  Despite the difference of opinion, he is a hig...  
2  It is better to stand for the truth than to sc...  
3  Jai Ravi Dai is always with the Nepali people ...  
4  Make a common agenda in a way that does not af...  

Done. All three YouTube splits translated.


In [ ]:
df= pd.read_csv("youtube_train_en.csv")
df[["text", "text_en"]].sample(20)


,text,text_en
452,ज्ञानेन्द् सरकार का पुर्खा हरू ले आर्जे को देश...,If the ancestors of the wise government had be...
105,ज्यान मार्ने पर्चन्ड अजई मान्छे मार्ने कुरा गरछ ।,The one who killed Perchand Ajai is talking ab...
606,यस्ता पुसिल्लाइ भाला नै हानु पर्छ साला कुकुर ह...,"Such a puppy should not be harmed by spears, b..."
1183,हत्यार बोकेर खुल्ला रूप मा हिड्ने मान्छे हरुला...,Anyone who walks openly carrying a murderer mu...
731,"100% सत्ये कुरा हो , अब देस को प्रधानमन्त्री र...","The truth is, the country must now be ruled by..."
639,गु खा अब देशी कुकुर ।,They are now native dogs.
650,वा के राम्रो बोली पत्रकार मित्र ज्यु छड्के सला...,Or a good speech journalist friend of Jewish p...
713,द्वारिकालाल जस्तो जनप्रतिनिधी हरु एउ टा हैन अब...,People like Dwarikalal are not the same now th...
835,रातो कोट लाउने सुपर ।,Super cool to wear a red coat .
1157,लोकतन्त्र मा नेता ले नागरिक ( अभियन्ता ) लाई ह...,"In a democracy, the leader is not going to tak..."
